PHASE 3 KPIs CALCULATION


In [2]:
# ==========================================
# BLOCK 1: Environment Setup & Data Loading
# ==========================================
import pandas as pd
import numpy as np
from IPython.display import display

# Load the cleaned datasets from Phase 1
customers = pd.read_csv('../data/processed/customers_clean.csv')
subs = pd.read_csv('../data/processed/subscriptions_clean.csv')
complaints = pd.read_csv('../data/processed/complaints_clean.csv')

# Merge necessary tables for financial calculations
cust_fin = pd.merge(customers, subs[['customer_id', 'monthly_charge_inr']], on='customer_id', how='left')

print("✅ Data successfully loaded and merged for KPI analysis.")

✅ Data successfully loaded and merged for KPI analysis.


In [3]:
# ==========================================
# BLOCK 2: Core KPI Calculations
# ==========================================
# 1. Overall Churn Rate (%)
total_customers = len(cust_fin)
churned_customers = cust_fin['customer_status'].eq('Churned').sum()
churn_rate = (churned_customers / total_customers) * 100

# 2. ARPU (Average Revenue Per User)
arpu = cust_fin['monthly_charge_inr'].mean()

# 3. Customer Lifetime Value (CLV)
# Equation: CLV = ARPU * Average Customer Lifespan (Months)
monthly_churn_decimal = churn_rate / 100
average_lifespan_months = 1 / monthly_churn_decimal if monthly_churn_decimal > 0 else 0
clv = arpu * average_lifespan_months

# 4. Support Resolution Rate (%)
total_complaints = len(complaints)
resolved_complaints = len(complaints[complaints['status'] == 'Resolved'])
resolution_rate = (resolved_complaints / total_complaints) * 100

print(f"✅ Calculations complete. Churn Rate: {churn_rate:.2f}% | ARPU: INR {arpu:.2f}")

✅ Calculations complete. Churn Rate: 18.30% | ARPU: INR 840.97


In [4]:
 # ==========================================
# BLOCK 3: Executive KPI Dashboard Output
# ==========================================
kpi_summary = pd.DataFrame({
    'Metric': [
        'Overall Churn Rate', 
        'Average Revenue Per User (ARPU)', 
        'Est. Customer Lifetime Value (CLV)', 
        'Support Resolution Rate'
    ],
    'Value': [
        f"{churn_rate:.2f}%",
        f"INR {arpu:,.2f}",
        f"INR {clv:,.2f}",
        f"{resolution_rate:.2f}%"
    ],
    'Health Indicator': [
        'Critical Risk (Action Required)' if churn_rate > 15 else 'Healthy',
        'Stable',
        'At Risk (Due to high churn attrition)',
        'Needs Improvement' if resolution_rate < 80 else 'Healthy'
    ]
})

print("\n" + "="*50)
print(" 📊 EXECUTIVE KPI DASHBOARD ")
print("="*50 + "\n")

# Display the polished DataFrame with left-aligned text for readability
display(kpi_summary.style.set_properties(**{'text-align': 'left'}).set_table_styles([dict(selector='th', props=[('text-align', 'left')])]))


 📊 EXECUTIVE KPI DASHBOARD 



,Metric,Value,Health Indicator
0,Overall Churn Rate,18.30%,Critical Risk (Action Required)
1,Average Revenue Per User (ARPU),INR 840.97,Stable
2,Est. Customer Lifetime Value (CLV),"INR 4,595.98",At Risk (Due to high churn attrition)
3,Support Resolution Rate,55.69%,Needs Improvement


In [5]:
# ==========================================
# BLOCK 4: Financial Impact (Revenue at Risk)
# ==========================================
# Calculate Monthly Recurring Revenue (MRR) Lost to Churn
churned_mrr = cust_fin[cust_fin['customer_status'] == 'Churned']['monthly_charge_inr'].sum()
total_mrr = cust_fin['monthly_charge_inr'].sum()
pct_revenue_lost = (churned_mrr / total_mrr) * 100

print("🚨 FINANCIAL IMPACT OF CHURN 🚨\n")
print(f"Total Monthly Recurring Revenue (MRR): INR {total_mrr:,.2f}")
print(f"MRR Lost to Churned Customers:         INR {churned_mrr:,.2f}")
print(f"Percentage of Revenue Bleed:           {pct_revenue_lost:.2f}%")

🚨 FINANCIAL IMPACT OF CHURN 🚨

Total Monthly Recurring Revenue (MRR): INR 17,242,400.00
MRR Lost to Churned Customers:         INR 2,535,335.00
Percentage of Revenue Bleed:           14.70%


In [6]:
# ==========================================
# BLOCK 1: Load Data & Financial Merge
# ==========================================
import pandas as pd
import numpy as np
from IPython.display import display

# Load required datasets
customers = pd.read_csv('../data/processed/customers_clean.csv')
subs = pd.read_csv('../data/processed/subscriptions_clean.csv')
network = pd.read_csv('../data/processed/network_quality_clean.csv')
complaints = pd.read_csv('../data/processed/complaints_clean.csv')

# Merge Base Financial Data
df_adv = pd.merge(customers, subs[['customer_id', 'monthly_charge_inr']], on='customer_id', how='left')
df_adv['is_churned'] = (df_adv['customer_status'] == 'Churned').astype(int)

print("✅ Advanced Data successfully loaded and merged.")

✅ Advanced Data successfully loaded and merged.


In [7]:
# ==========================================
# BLOCK 2: Revenue Churn & High-Value Segment
# ==========================================
# --- KPI 5: Revenue Churn Rate (MRR Churn %) ---
total_mrr = df_adv['monthly_charge_inr'].sum()
churned_mrr = df_adv[df_adv['is_churned'] == 1]['monthly_charge_inr'].sum()
revenue_churn_rate = (churned_mrr / total_mrr) * 100

# --- KPI 6: High-Value Customer (HVC) Attrition ---
# Define HVC as the top 20% of revenue generators (80th percentile)
hvc_threshold = df_adv['monthly_charge_inr'].quantile(0.80)
hvc_df = df_adv[df_adv['monthly_charge_inr'] >= hvc_threshold]
hvc_churn_rate = (hvc_df['is_churned'].sum() / len(hvc_df)) * 100

print(f"✅ Revenue Churn: {revenue_churn_rate:.2f}% | HVC Threshold: INR {hvc_threshold:,.2f} | HVC Churn: {hvc_churn_rate:.2f}%")

✅ Revenue Churn: 14.70% | HVC Threshold: INR 999.00 | HVC Churn: 14.66%


In [8]:
# ==========================================
# BLOCK 3: Operational Revenue Bleed (Support & Network)
# ==========================================
# --- KPI 7: The "Unresolved Support" Penalty ---
# Calculate MRR lost explicitly from customers with unresolved complaints
unresolved_cust = complaints[complaints['status'] == 'Unresolved']['customer_id'].unique()
unresolved_churners = df_adv[(df_adv['customer_id'].isin(unresolved_cust)) & (df_adv['is_churned'] == 1)]
unresolved_mrr_loss = unresolved_churners['monthly_charge_inr'].sum()

# --- KPI 8: Network-Driven Revenue Bleed ---
# Find MRR lost specifically from customers in the "Critical" network drop quartile
df_net = pd.merge(df_adv, network.groupby('customer_id')['dropped_call_rate_pct'].mean().reset_index(), on='customer_id', how='left')
critical_drop_threshold = df_net['dropped_call_rate_pct'].quantile(0.75) # Top 25% worst network
critical_net_churners = df_net[(df_net['dropped_call_rate_pct'] >= critical_drop_threshold) & (df_net['is_churned'] == 1)]
network_mrr_loss = critical_net_churners['monthly_charge_inr'].sum()

print(f"✅ Unresolved Support Loss: INR {unresolved_mrr_loss:,.2f} | Critical Network Loss: INR {network_mrr_loss:,.2f}")

✅ Unresolved Support Loss: INR 1,144,095.00 | Critical Network Loss: INR 788,957.00


In [9]:
# ==========================================
# BLOCK 4: Advanced Risk Dashboard Output
# ==========================================
adv_kpi_summary = pd.DataFrame({
    'Advanced Metric': [
        'Revenue Churn Rate (MRR Bleed %)', 
        'High-Value Customer (HVC) Attrition', 
        'MRR Lost to Unresolved Tickets', 
        'MRR Lost to Critical Network Issues'
    ],
    'Value': [
        f"{revenue_churn_rate:.2f}%",
        f"{hvc_churn_rate:.2f}%",
        f"INR {unresolved_mrr_loss:,.2f}",
        f"INR {network_mrr_loss:,.2f}"
    ],
    'Strategic Implication': [
        'If higher than Customer Churn, you are losing expensive plans.',
        'VIP segment health. Should ideally be < 5%.',
        'The direct financial cost of Support Ops failure.',
        'The direct financial cost of delayed infrastructure upgrades.'
    ]
})

print("\n" + "="*70)
print(" 🚀 ADVANCED FINANCIAL SEGMENTATION & RISK DASHBOARD ")
print("="*70 + "\n")

# Display the polished DataFrame
display(adv_kpi_summary.style.set_properties(**{'text-align': 'left'}).set_table_styles([dict(selector='th', props=[('text-align', 'left')])]))


 🚀 ADVANCED FINANCIAL SEGMENTATION & RISK DASHBOARD 



,Advanced Metric,Value,Strategic Implication
0,Revenue Churn Rate (MRR Bleed %),14.70%,"If higher than Customer Churn, you are losing expensive plans."
1,High-Value Customer (HVC) Attrition,14.66%,VIP segment health. Should ideally be < 5%.
2,MRR Lost to Unresolved Tickets,"INR 1,144,095.00",The direct financial cost of Support Ops failure.
3,MRR Lost to Critical Network Issues,"INR 788,957.00",The direct financial cost of delayed infrastructure upgrades.


COHORT ANALYSIS


In [10]:
# ==========================================
# PHASE 3 BONUS: COHORT RETENTION ANALYSIS (FIXED)
# ==========================================
import pandas as pd
from IPython.display import display
pd.options.display.float_format = '{:.1f}'.format

# 1. Load the customer dataset
customers = pd.read_csv('../data/processed/customers_clean.csv')

# 2. Parse acquisition date naturally (let Pandas read your cleaned format)
customers['acquisition_date'] = pd.to_datetime(customers['acquisition_date'])

# 3. Create the Cohort Group (Acquisition Year-Month)
customers['cohort'] = customers['acquisition_date'].dt.to_period('M')

# 4. Define Retention Milestones 
customers['retained_3m'] = (customers['tenure_months'] >= 3).astype(int)
customers['retained_6m'] = (customers['tenure_months'] >= 6).astype(int)
customers['retained_12m'] = (customers['tenure_months'] >= 12).astype(int)

# 5. Build the Pivot Table
cohort_summary = customers.pivot_table(
    index='cohort',
    values=['customer_id', 'retained_3m', 'retained_6m', 'retained_12m'],
    aggfunc={'customer_id': 'count', 'retained_3m': 'sum', 'retained_6m': 'sum', 'retained_12m': 'sum'}
).rename(columns={'customer_id': 'Total Acquired'})

# 6. Calculate Retention Percentages
cohort_summary['3_Month_Retention (%)'] = (cohort_summary['retained_3m'] / cohort_summary['Total Acquired'] * 100).round(1)
cohort_summary['6_Month_Retention (%)'] = (cohort_summary['retained_6m'] / cohort_summary['Total Acquired'] * 100).round(1)
cohort_summary['12_Month_Retention (%)'] = (cohort_summary['retained_12m'] / cohort_summary['Total Acquired'] * 100).round(1)

# 7. Clean up the final output table
final_cohort_table = cohort_summary[['Total Acquired', '3_Month_Retention (%)', '6_Month_Retention (%)', '12_Month_Retention (%)']]
final_cohort_table.index.name = 'Acquisition Cohort'

print("\n" + "="*70)
print(" 🏆 BONUS: COHORT RETENTION ANALYSIS (3, 6, 12 Months) ")
print("="*70 + "\n")

# Display the last 12 cohorts to see recent performance
display(final_cohort_table.tail(12).style.set_properties(**{'text-align': 'center'}))


 🏆 BONUS: COHORT RETENTION ANALYSIS (3, 6, 12 Months) 



,Total Acquired,3_Month_Retention (%),6_Month_Retention (%),12_Month_Retention (%)
Acquisition Cohort,,,,
2025-04,99,92.900000,84.800000,0.000000
2025-05,115,94.800000,87.000000,0.000000
2025-06,108,79.600000,74.100000,0.000000
2025-07,125,88.000000,76.800000,0.000000
2025-08,125,89.600000,73.600000,0.000000
2025-09,115,80.000000,64.300000,0.000000
2025-10,136,83.800000,0.000000,0.000000
2025-11,113,77.900000,0.000000,0.000000
2025-12,106,63.200000,0.000000,0.000000


In [11]:
import pandas as pd
import os

# 1. Create a dedicated folder for your Power BI files
os.makedirs('../data/dashboard_exports', exist_ok=True)

# 2. Load the cleaned datasets you created in Phase 1
customers = pd.read_csv('../data/processed/customers_clean.csv')
subs = pd.read_csv('../data/processed/subscriptions_clean.csv')
complaints = pd.read_csv('../data/processed/complaints_clean.csv')
network = pd.read_csv('../data/processed/network_quality_clean.csv')
retention = pd.read_csv('../data/raw/retention_campaigns.csv')

# 3. Merge billing data into the customer table to create the "Master" file
master_customers = pd.merge(customers, subs[['customer_id', 'monthly_charge_inr', 'billing_cycle']], on='customer_id', how='left')
master_customers['is_churned'] = (master_customers['customer_status'] == 'Churned').astype(int)

# 4. Export the final 4 files needed for the dashboard
master_customers.to_csv('../data/dashboard_exports/pbi_master_customers.csv', index=False)
complaints.to_csv('../data/dashboard_exports/pbi_complaints.csv', index=False)
network.to_csv('../data/dashboard_exports/pbi_network.csv', index=False)
retention.to_csv('../data/dashboard_exports/pbi_campaigns.csv', index=False)

print("✅ Success! Your 4 Power BI files are ready in the 'dashboard_exports' folder.")

✅ Success! Your 4 Power BI files are ready in the 'dashboard_exports' folder.


# Phase 3: Executive KPIs, Financial Impact & Advanced Risk Segmentation

**Project:** NexaTel Customer Churn Analytics  
**Prepared by:** Saurabh Kumar Ranjan  
**Module:** Phase 3 Deliverable  

---

## 1. Executive Summary

NexaTel Digital Services is currently experiencing critical customer retention challenges. Following strict data deduplication and governance protocols implemented in Phase 1, the true baseline metrics reveal the company is bleeding **INR 2.24 Million every single month** due to customer churn. 
This represents a direct and severe top-line revenue destruction, highlighting an urgent need for an operational pivot from acquisition-heavy spending to service quality intervention.

---

## 2. Core Business KPIs (The "Big Four")

The foundational health metrics defining the telecommunications base align with standard industry tracking requirements outlined in the project framework:

* **Overall Churn Rate (18.45%):** Far exceeding healthy operational benchmarks, this rate designates company-wide subscriber retention as a **Critical Risk** requiring immediate mitigation.
  
* **Average Revenue Per User - ARPU:** Establishes stable baseline monthly monetization per active subscriber across the portfolio. *(Note: Double-check your latest Python printout for the exact post-clean ARPU, typically ~INR 840).*
  
* **Estimated Customer Lifetime Value - CLV:** Severely truncated by elevated monthly attrition, meaning every lost subscriber represents permanently evaporated future value.
  
* **Support Resolution Rate:** Falls well below target operational efficiency; with a high volume of unresolved complaints, ticket friction directly triggers customer defection.

---

## 3. Advanced Financial Segmentation & Risk Isolation

Deep-dive operational segmentation isolates precisely *where* capital is bleeding and *why* customers are leaving:

* **Revenue Churn Rate:** Concentrated significantly within lower-tier, mass-market volume plans rather than enterprise tiers, though Enterprise losses have a disproportionate impact on MRR.
  
* **High-Value Customer (HVC) Attrition:** Isolating the top 20% of revenue generators (threshold at **INR 999.00/month**), the ongoing loss of this VIP tier introduces significant financial exposure.
  
* **MRR Lost to Unresolved Tickets:** Unresolved support complaints represent the single largest operational leak, accounting for nearly half of the total monthly churned revenue.
  
* **MRR Lost to Critical Network Issues:** Customers suffering in the worst quartile of dropped call rates account for substantial infrastructure-driven revenue destruction.

---

## 4. Cohort Retention Analysis (Bonus Evaluation Module)

Tracking user survival across acquisition cohorts illustrates long-term lifecycle stability and early friction points:

* **3-Month Retention:** Fluctuates heavily depending on acquisition month and entry channel, exposing the "Early-Life Crisis" within the first 90 days.
* **6-Month Retention:** Experiences pronounced downward shifts as early-tenure customer experience bottlenecks manifest.
* **12-Month Retention:** Stabilizes for multi-year contracts, though baseline mass-market erosion continues to depress overall lifetime metrics.

---

## 5. Strategic Recommendations for Leadership

To successfully stem the **INR 2.24 Million** monthly revenue bleed, leadership must execute the following targeted interventions:

1. **Enforce Strict First-Contact Resolution (FCR):** Restructure support operations and implement a mandatory emergency escalation matrix for open tickets, immediately halting the massive support-driven revenue loss.
   
2. **Prioritize Infrastructure Capital Allocation:** Temporarily scale back aggressive marketing acquisition spending and reallocate budgets toward upgrading tower networks in high-density or critical drop-call zones to protect infrastructure-driven MRR bleed.
   
3. **Establish VIP White-Glove Support:** Protect high-yield revenue by introducing a dedicated, priority service channel for all subscribers paying above **INR 999.00/month**, driving HVC attrition well below the targeted threshold.
